# Assignment 1

## Question 1
Suppose you are a portfolio manager and you are going to use derivatives to construct certain portfolio. The current stock price for AAPL is $171.01 per share, at 10 am, March 8, 2019. You want to sell 1 unit of European Call on AAPL, with strike of $180 and maturity of 1 year. 

Suppose the annualized interest rate is 3%, the annualized drift for AAPL is 5%, no dividend, and the annualized volatility is 10%. In order to hedge the potential risk of your option, you are going to calculate several Greeks based on Black-Merton-Scholes model. 

In [2]:
from math import log, sqrt, exp
from scipy import stats

### Helper Functions

In [ ]:
def bsm_call_price(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float
):
    """
    parameters
    =====
    S0: spot price
    K: strike price
    T: time to maturity (years)
    r: risk free interest rate
    q: continuous dividend yield
    sigma: volatility
    """
    S0 = float(S0)
    d1 = (log(S0 / K) + ((r - q) + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - (sigma * sqrt(T))
    # stats.norm.cdf --> cumulative distribution function
    value = (S0 * stats.norm.cdf(d1, 0.0, 1.0) - K * exp(-r * T) * stats.norm.cdf(d2, 0.0, 1.0))

    return value

In [14]:
def bsm_call_delta(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float
):
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    value = exp(-q * T) * stats.norm.cdf(d1, 0.0, 1.0)

    return value

In [16]:
def bsm_call_gamma(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float
):
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    value = exp(-q * T) * stats.norm.pdf(d1, 0.0, 1.0)/ (S0 * sigma * sqrt(T))

    return value

In [18]:
def bsm_call_vega(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float
):
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    value = S0 * sqrt(T) * stats.norm.pdf(d1, 0.0, 1.0)

    return value

In [20]:
def bsm_call_theta(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float
):
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - (sigma * sqrt(T))
    value = - ((S0 * stats.norm.pdf(d1, 0.0, 1.0) * sigma)/(2 * sqrt(T))) - (r * K * exp(-r * T) * stats.norm.cdf(d2, 0.0, 1.0))

    return value

## Results

In [13]:
# 1. call option price
call_option_price = bsm_call_price(171.01, 180.0, 1.0, 0.03, 0.0, 0.1)

print(f'Call Option Price: {round(call_option_price, 4)}')

Call Option Price: 5.2122


In [15]:
# 2. call option delta
call_option_delta = bsm_call_delta(171.01, 180.0, 1.0, 0.03, 0.0, 0.1)

print(f'Call Option Delta: {round(call_option_delta, 4)}')

Call Option Delta: 0.4355


In [17]:
# 3. call option gamma
call_option_gamma = bsm_call_gamma(171.01, 180.0, 1.0, 0.03, 0.0, 0.1)

print(f'Call Option Gamma: {round(call_option_gamma, 4)}')

Call Option Delta: 0.023


In [19]:
# 4. call option vega
call_option_vega = bsm_call_vega(171.01, 180.0, 1.0, 0.03, 0.0, 0.1)

print(f'Call Option Vega: {round(call_option_vega, 4)}')

Call Option Vega: 67.3299


In [21]:
# 5. call option theta
call_option_theta = bsm_call_theta(171.01, 180.0, 1.0, 0.03, 0.0, 0.1)

print(f'Call Option Theta: {round(call_option_theta, 4)}')

Call Option Theta: -5.4445


Question 6
After selling the option, you want to do Delta-Hedge immediately. So you are currently buying Delta shares of AAPL, where Delta is your calculation result in Question 1. Calculate the value of your portfolio (i.e. the net cashflow) of selling one unit of call, and buying Delta shares of AAPL, at 10 am, March 8, 2019. For example, if you construct your portfolio by borrowing $10, then the answer is -10. Same for following questions.

Question 7
Suppose you are doing a monthly Delta-Hedge, that is, you re-hedge per month. After one month, at 10 am, April 8, 2019, the stock price is $180.2, and you want to re-hedge right now. Calculate the additional amount of shares of AAPL you should buy (positive for buying, negative for selling) at 10 am, April 8, 2019, in order to re-hedge.

Question 8
Calculate the change of your portfolio value (i.e. the net cashflow). Note that your answer should be in $ with new portfolio value minus previous value, in present value at 10 am, April 8, 2019.

In [22]:
# 6. net cash flow = cash from call option sell - cash going out on purchase
net_cash_flow = call_option_price - (171.01 * call_option_delta)

print(f'Portfolio Value: {round(net_cash_flow, 4)}')

Portfolio Value: -69.2653


In [25]:
# 7. in a month, we need to caclulate new BSM metrics
# S0 = 180.2, K remains the same, T is 1 year - 1 month (11/12)
call_option_delta_in_1_month = bsm_call_delta(180.2, 180.0, 11/12, 0.03, 0.0, 0.1)
print(f'Additional Shares: {round(call_option_delta_in_1_month-call_option_delta, 4)}')

Additional Shares: 0.2001


In [37]:
# 8. change in portfolio value
# first calculate new call option price
call_option_price_in_1_month = bsm_call_price(180.2, 180.0, 11/12, 0.03, 0.0, 0.1)
value_shares_held = (180.2 * call_option_delta)

# net porfolio = -calloption price + value of shares held + compounded bank debt
change_in_portfolio_value = -(call_option_price_in_1_month) + value_shares_held + (net_cash_flow * exp(0.03 * 1/12))
print(f'Change In Portfolio Value: {round(change_in_portfolio_value, 4)}')

Change In Portfolio Value: -0.5836


In [35]:
call_option_price_in_1_month

np.float64(9.510828023555788)